# Real opamps

```{admonition} How To Work
:class: important
The **Preparatory Homework (BAS)** part of this manual should be completed **individually**.

The **Practicum (IC)** part should be completed **in pairs**. Bias and offset numbers are small (millivolts and nanoamps); take turns reading the meter so you spot transcription mistakes early.

Throughout this manual, keep your derivations, datasheet snippets, scope screenshots, and short reflections in your notepad. The comparison tables in the Compare and Conclude section should be filled in your notepad and included in the onepager report for this lab.
```


## Preparatory Homework

### Background
```{admonition} Preparation
:class: tip
You should have completed [Manual 4.1](4.1_opamp_basics.ipynb) before starting this one. We will reuse the non-inverting and inverting circuits you already know how to build, and add three more measurements that probe the **non-ideal** parts of a real opamp.

Re-read the textbook chapter on the non-ideal opamp before you start.

* [Textbook chapter 8: real opamps](../../../theory_part/8_opamp2.md)

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">🚩 TODO: replace with the published textbook URL or anchor for chapter 8</span>
```

In Manual 4.1 the **golden rules** were enough:

1. The two opamp inputs sit at the same voltage (the virtual short).
2. No current enters the opamp inputs.
3. The output behaves like an ideal voltage source with zero output impedance.
4. The gain is independent of frequency.

A real opamp obeys all four rules **only approximately**. Today you will measure five imperfections and apply them to a circuit (the integrator) where almost every one of them shows up.

```{figure} images/OPAMP_with_Bias_and_offset.svg
---
name: fig-non-ideal-opamp-model
height: 320px
---
Non-ideal opamp model: an ideal opamp with two bias-current sources (one per input) and an offset voltage source in series with one input.
```

#### The five non-idealities

* **Input bias current** ($I_{bias}$). Each input draws a small DC current to bias its first transistor stage. For the LM358 it is tens of nanoamps (typical $45\,\mathrm{n A}$). The current must have a path to ground at every input, otherwise the opamp will drift to one of its rails.
* **Input offset voltage** ($V_{offset}$). The two input transistors are not perfectly matched. Even with both inputs tied to ground, the opamp behaves as if there were a small voltage source (a few millivolts) in series with one input. In a high-gain amplifier this offset is amplified by the same factor as the signal.
* **Gain-bandwidth product** (GBW). The open-loop gain is huge at DC (about $10^5$) but it falls off at $-20\,\mathrm{dB/decade}$ above a low corner. The product $\text{gain} \times \text{bandwidth}$ is approximately constant. For the LM358, GBW $\approx 1\,\mathrm{MHz}$.

  For the **non-inverting** amplifier, the closed-loop bandwidth is

  $$
  \mathrm{BW} = \frac{\mathrm{GBW}}{1 + R_a/R_b} = \frac{\mathrm{GBW}}{G}.
  $$

  The "+1" is small once the gain is large but matters for low-gain amplifiers.

  ```{figure} images/gbw_bode.svg
  ---
  name: fig-gbw-bode
  height: 320px
  ---
  The gain-bandwidth trade-off as a Bode plot. The open-loop gain (dashed) is huge at DC but falls at $-20\,\mathrm{dB/decade}$, crossing $0\,\mathrm{dB}$ at the GBW of about $1\,\mathrm{MHz}$. Each closed-loop gain is flat until it runs into the open-loop curve; the dots mark the $-3\,\mathrm{dB}$ points, which sit at $\mathrm{GBW}/G$. Ten times more gain buys ten times less bandwidth.
  ```

* **Slew rate** (SR). The output cannot change faster than $\mathrm{SR}$ V/microsecond, regardless of the gain or the bandwidth. For the LM358, SR $\approx 0.5\,\mathrm{V/\mu s}$. A square wave whose ideal output exceeds this slope comes out as a trapezoid; a sine whose peak slope $2\pi f \hat U$ exceeds the slew rate comes out as a triangle.

  The datasheet name for the highest frequency at which the opamp can still deliver its **full rated output swing** is the **full power bandwidth**:

  $$
  f_{FPBW} = \frac{\mathrm{SR}}{2\pi \hat U_{max}}.
  $$

  You will compute it in [Task A4](Task_A4_4_2) and check it against your measured slew rate in [Task I2](Task_I2_4_2).

* **Maximum output current** ($I_{out,max}$). Golden rule 3 promises an ideal voltage source, but the output stage can only source or sink a limited current, about $20\,\mathrm{m A}$ for the LM358 (the exact limit differs between sourcing and sinking, and the chip is internally short-circuit protected). With a small load resistor, the output current hits this limit before the voltage reaches the value the feedback demands, and the waveform clips at $\pm I_{out,max} R_L$ instead of at the supply rails.

```{figure} images/slew_rate_square.svg
---
name: fig-slew-rate-square
height: 220px
---
The slew-rate limit turns every edge into a ramp of fixed slope $\pm\mathrm{SR}$ ($0.5\,\mathrm{V/\mu s}$ here). At $2\,\mathrm{kHz}$ the ramps are barely visible and the output still looks square. At $10\,\mathrm{kHz}$ the ramps eat a large part of each half-period and the square becomes a trapezoid. At $50\,\mathrm{kHz}$ a ramp cannot finish before the input flips again, so the output collapses into a triangle that only reaches $\pm 2.5\,\mathrm{V}$ instead of $\pm 5\,\mathrm{V}$.
```

#### A quick refresher on dB and decade

You will be reading off bandwidths in this manual, which means handling logarithmic scales.

* $1\,\mathrm{decade}$ is a factor of 10 in frequency.
* $1\,\mathrm{octave}$ is a factor of 2.
* A gain expressed in $\mathrm{dB}$ is $20 \log_{10}(|G|)$. So $G = 10$ is $20\,\mathrm{dB}$ and $G = 100$ is $40\,\mathrm{dB}$.
* The **$-3\,\mathrm{dB}$ point** is the frequency at which the closed-loop gain has dropped to $|G|/\sqrt{2} \approx 0.71\,|G|$. Above the $-3\,\mathrm{dB}$ point the amplifier no longer follows the input cleanly.

#### The chip you are going to characterise

Every measurement in this manual uses the **LM358P**, one of the two opamps in your kit. It is a dual opamp: two identical opamps share one DIP-8 package, and you will use opamp 1, exactly as in Manual 4.1. Its bipolar inputs draw a bias current of tens of nanoamps, large enough to measure with bench instruments (the kit's other opamp, the TL072CP, has JFET inputs whose picoamp bias current would be invisible in these measurements).

* [LM358 datasheet (cached)](../../../ei_helpers/datasheet_pdfs/LM358.pdf)

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">🚩 TODO: download the LM358 datasheet, save it under ei_helpers/datasheet_pdfs/, verify this link, and check the typical values used throughout this manual (Voffset 2 mV, Ibias 45 nA, GBW 1 MHz, SR 0.5 V/us, output current limit 20 mA) against it</span>


### Anticipate

The six tasks below let you predict every measurement before you take it. In most of them you do not just verify a given circuit, you **design**: you choose component values against a spec, the way you would in a real instrument. The big skill of this manual is not the algebra, it is the reasoning about which non-ideality dominates in a given circuit. That is exactly what design engineers do every day.


(Task_A1_4_2)=
#### Task A1: Choose $R_a$ to isolate $V_{offset}$
```{admonition} Estimated time: 10 min
:class: estimated-time no-content
```

You will use a non-inverting amplifier with a single feedback resistor $R_f$ tied to the inverting input and a resistor $R_a$ from the inverting input to ground. The non-inverting input is grounded.

```{figure} images/bias_offset_test_circuit.svg
---
name: fig-bias-offset-test-circuit
height: 320px
---
Test circuit for bias and offset measurement: $R_f = 1\,\mathrm{M}\Omega$ is fixed; $R_a$ is the parameter you choose.
```

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">🚩 TODO: draw and add images/bias_offset_test_circuit.svg (LM358 opamp 1, +/-12 V, R_f = 1M in feedback, R_a from inverting to GND, non-inverting input to GND through a small R)</span>

The output voltage of this circuit is approximately

$$
U_{out} \approx \left(1 + \frac{R_f}{R_a}\right) V_{offset} + I_{bias} R_f
$$

(where $I_{bias}$ is the bias current of the inverting input).

1. With $R_f = 1\,\mathrm{M}\Omega$ fixed, choose an $R_a$ that makes the **first term dominant**: that is, the offset-voltage term is much larger than the bias-current term. Justify your choice with a one-line inequality, using LM358 typical specs ($V_{offset} \approx 2\,\mathrm{m V}$, $I_{bias} \approx 45\,\mathrm{n A}$).

2. With your chosen $R_a$, compute the predicted $U_{out}$. Is it large enough to be read cleanly with a $3\,\mathrm{1/2}$-digit DMM (resolution $\sim 1\,\mathrm{m V}$ on the 2 V range)?

```{admonition} Check your answer
:class: answer, dropdown
For a small $R_a$ (say $1\,\mathrm{k}\Omega$), the gain factor on $V_{offset}$ is $1 + 1000/1 \approx 1000$, so the offset term contributes $\sim 2\,\mathrm{V}$, while the bias term is $45\,\mathrm{n A} \cdot 1\,\mathrm{M}\Omega = 45\,\mathrm{m V}$. Offset dominates by $\sim 44\times$. The DMM reads $\sim 2\,\mathrm{V}$ comfortably on the 2 V range.
```

(Task_A2_4_2)=
#### Task A2: Choose $R_a$ to isolate $I_{bias}$
```{admonition} Estimated time: 10 min
:class: estimated-time no-content
```

With the same test circuit, choose an $R_a$ that makes the **second term dominant**: the bias-current term is much larger than the offset term. Justify with the same kind of inequality.

For your chosen $R_a$, predict $U_{out}$ once again.

```{admonition} Check your answer
:class: answer, dropdown
For a large $R_a$ (say $10\,\mathrm{M}\Omega$, so the offset gain is $1 + 1\,\mathrm{M}/10\,\mathrm{M} = 1.1$), the offset term contributes only $\sim 2\,\mathrm{m V}$, while the bias term is $45\,\mathrm{n A} \cdot 1\,\mathrm{M}\Omega = 45\,\mathrm{m V}$. Bias dominates by $\sim 20\times$. The DMM reads $\sim 47\,\mathrm{m V}$, comfortable on the 200 mV range.
```

This is exactly the engineering skill you want to take away from this manual: pick the test condition that **isolates** the parameter you want to measure.

(Task_A3_4_2)=
#### Task A3: Design three amplifiers and predict their bandwidth
```{admonition} Estimated time: 15 min
:class: estimated-time no-content
```

You will characterise the LM358 gain-bandwidth product with three non-inverting amplifiers of widely spaced gains: $G \approx 2$, $G \approx 11$, and $G \approx 101$.

1. **Design them.** Choose $R_a$ (feedback) and $R_b$ (inverting input to ground) for each gain, using E12 values. Respect two constraints and justify your choice against both in one line each:
   * The feedback network loads the output with $R_a + R_b$, and the output current is limited (see Background). Do not pick values so small that the feedback network alone draws a large fraction of $I_{out,max}$ at full swing.
   * The bias current flowing through the feedback network produces an output error of roughly $I_{bias} \cdot R_a$. Do not pick values so large that this error becomes visible next to your signal.

2. Using the GBW formula from the Background, predict the $-3\,\mathrm{dB}$ frequency for each amplifier. Fill in the prediction column of the table below; the simulation and measurement columns are for [Task S2](Task_S2_4_2) and [Task I2](Task_I2_4_2).

| $G$ (set) | predicted $\mathrm{BW}$ | simulated $\mathrm{BW}$ | measured $\mathrm{BW}$ | $G \cdot \mathrm{BW}$ |
|---|---|---|---|---|
| 2 |  |  |  |  |
| 11 |  |  |  |  |
| 101 |  |  |  |  |

**Important:** the textbook writes the formula as $\mathrm{BW} = \mathrm{GBW}/G$, but the strict formula for the non-inverting amplifier is $\mathrm{BW} = \mathrm{GBW}/(1 + R_a/R_b)$. The "$1+$" matters for the $G = 2$ row. See [textbook chapter 8](../../../theory_part/8_opamp2.md).

```{admonition} Check your answer
:class: answer, dropdown
A solid middle-of-the-road choice is $R_b = 1\,\mathrm{k}\Omega$ everywhere, with $R_a = 1\,\mathrm{k}\Omega$, $10\,\mathrm{k}\Omega$, and $100\,\mathrm{k}\Omega$. At full swing ($10\,\mathrm{V}$) the heaviest feedback network ($G = 2$) draws $10\,\mathrm{V}/2\,\mathrm{k}\Omega = 5\,\mathrm{m A}$, well below the $20\,\mathrm{m A}$ limit; the worst bias error is $45\,\mathrm{n A} \cdot 100\,\mathrm{k}\Omega = 4.5\,\mathrm{m V}$, invisible next to a volt-scale signal. The predicted bandwidths are then $500\,\mathrm{kHz}$, $91\,\mathrm{kHz}$, and $9.9\,\mathrm{kHz}$. The practicum text assumes these values; your own picks are fine if they meet both constraints.
```

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">🚩 TODO: replace with the published anchor for the GBW section in chapter 8</span>

(Task_A4_4_2)=
#### Task A4: Predict the slew-limited frequency and the full power bandwidth
```{admonition} Estimated time: 15 min
:class: estimated-time no-content
```

For an output sine wave $U_{out}(t) = \hat U \sin(2\pi f t)$, the maximum slope is $2\pi f \hat U$. The opamp can keep up with this slope only if $2\pi f \hat U \le \mathrm{SR}$.

1. For $\hat U = 5\,\mathrm{V}$ and the LM358's $\mathrm{SR} = 0.5\,\mathrm{V/\mu s}$, what is the largest $f$ at which the output is **not** slew-limited?
2. Sketch what you expect the output to look like at $f = 10$ times that value: still a sine, a triangle, a trapezoid?
3. Compare with the bandwidth of the $G = 11$ amplifier from Task A3. Which limit kicks in first as you raise the frequency: the bandwidth or the slew rate?
4. With $\pm 12\,\mathrm{V}$ supplies the LM358 can swing to about $\hat U_{max} = 10\,\mathrm{V}$. Compute the **full power bandwidth** $f_{FPBW} = \mathrm{SR}/(2\pi \hat U_{max})$. This is the number a datasheet quotes for large-signal operation; look it up in the LM358 datasheet and check that your value is in the same range.

```{admonition} Check your answer
:class: answer, dropdown
$f_{\max} = \mathrm{SR} / (2\pi \hat U) = 0.5 \cdot 10^6 / (2\pi \cdot 5) \approx 16\,\mathrm{kHz}$. At ten times that, the output cannot keep up: the sine becomes a triangle. The $G = 11$ bandwidth is about $1\,\mathrm{MHz}/11 \approx 90\,\mathrm{kHz}$, well above the slew limit at this amplitude. So the slew rate is the active limit at $\hat U = 5\,\mathrm{V}$. The full power bandwidth is $0.5 \cdot 10^6/(2\pi \cdot 10) \approx 8\,\mathrm{kHz}$: above $8\,\mathrm{kHz}$ this opamp cannot deliver a full-swing sine at all, no matter how the gain is set.
```


(Task_A5_4_2)=
#### Task A5: Choose the minimum load resistance
```{admonition} Estimated time: 10 min
:class: estimated-time no-content
```

Golden rule 3 says the output is an ideal voltage source. The Background gives the real limit: the LM358 can source or sink at most about $I_{out,max} \approx 20\,\mathrm{m A}$.

1. You want an undistorted sine of amplitude $\hat U = 10\,\mathrm{V}$ at the output. What is the minimum load resistance $R_{L,min}$ the amplifier can drive? Pick the nearest safe E12 value.
2. Predict the output waveform if you connect $R_L = 100\,\Omega$ instead and still ask for the $10\,\mathrm{V}$ sine. At what amplitude does it clip, and how would you tell this kind of clipping apart from clipping at the supply rails?
3. A classmate suggests driving an $8\,\Omega$ loudspeaker directly from the opamp output. In one line, why is this hopeless?

```{admonition} Check your answer
:class: answer, dropdown
$R_{L,min} = \hat U / I_{out,max} = 10\,\mathrm{V} / 20\,\mathrm{m A} = 500\,\Omega$; the nearest safe E12 value is $560\,\Omega$. With $R_L = 100\,\Omega$ the output clips at about $\pm I_{out,max} R_L = \pm 2\,\mathrm{V}$. Current clipping moves when you change $R_L$; rail clipping always sits near $\pm 10\,\mathrm{V}$ whatever the load. The loudspeaker would ask for $10\,\mathrm{V}/8\,\Omega > 1\,\mathrm{A}$, roughly sixty times the available current.
```


(Task_A6_4_2)=
#### Task A6: Design the integrator
```{admonition} Estimated time: 15 min
:class: estimated-time no-content
```

```{figure} images/integrator.svg
---
name: fig-integrator
height: 280px
---
Inverting integrator: input through $R$, capacitor $C$ in feedback.
```

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">🚩 TODO: draw and add images/integrator.svg (LM358 opamp 1, R from input to inverting, C in feedback, non-inverting input to GND)</span>

For the integrator in {numref}`fig-integrator`, the output is

$$
U_{out}(t) = -\frac{1}{RC} \int_0^t U_{in}(\tau)\, d\tau + U_{out}(0).
$$

The input will be a $50\,\mathrm{Hz}$ square wave that switches between $-2\,\mathrm{V}$ and $+2\,\mathrm{V}$. Your job is to **choose $R$ and $C$**, not to verify given ones.

1. While $U_{in}$ is constant at $+2\,\mathrm{V}$, show that the output is a ramp with slope $\mathrm{d}U_{out}/\mathrm{d}t = -U_{in}/(RC)$, and that over one half-period $T/2$ the output changes by $U_{in} \cdot (T/2)/(RC)$. Sketch one full period of $U_{in}(t)$ and $U_{out}(t)$ on the same time axis: the output is a triangle.
2. Choose $R$ and $C$ (E12 values, $R$ between $1\,\mathrm{k}\Omega$ and $1\,\mathrm{M}\Omega$, $C$ between $1\,\mathrm{n F}$ and $1\,\mathrm{\mu F}$) so that the output triangle has a peak-to-peak amplitude of roughly $4\,\mathrm{V}$: large enough to read cleanly on the scope, far enough from the $\pm 12\,\mathrm{V}$ rails.
3. What peak-to-peak amplitude would the seemingly innocent choice $R = 10\,\mathrm{k}\Omega$, $C = 100\,\mathrm{n F}$ give? Would it fit between the rails?

For the integrator derivation, see [textbook chapter 7, section on the integrator](../../../theory_part/7_opamp1.md).

```{admonition} Check your answer
:class: answer, dropdown
The peak-to-peak amplitude is $U_{in} \cdot (T/2)/(RC) = 2 \cdot 0.01/RC$. A $4\,\mathrm{V}$ target asks for $RC = 5\,\mathrm{ms}$; the E12 pair $R = 47\,\mathrm{k}\Omega$, $C = 100\,\mathrm{n F}$ gives $RC = 4.7\,\mathrm{ms}$ and $4.3\,\mathrm{V}$ peak-to-peak. The tasks below use these values; your own pick is fine as long as the amplitude comes out in the same range. The innocent choice $R = 10\,\mathrm{k}\Omega$, $C = 100\,\mathrm{n F}$ gives $RC = 1\,\mathrm{ms}$ and a demanded $20\,\mathrm{V}$ peak-to-peak: the opamp would clip near the rails before finishing each ramp.
```

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">🚩 TODO: replace with the published anchor for the integrator section in chapter 7</span>


### Simulate

All simulations in this manual are done in **LTspice**. You do not need to download any opamp model: you will build your own simulated LM358 from LTspice's built-in **UniversalOpAmp2**.

```{attention}
Place a **UniversalOpAmp2** (component browser, Opamps folder). Right click the symbol to open its parameter dialog and change the defaults to the LM358 values used throughout this manual (leave `Rail`, the noise parameters, and `Rin` at their defaults):

* `Avol = 100k` (open-loop DC gain, $100\,\mathrm{dB}$)
* `GBW = 1Meg` (gain-bandwidth product)
* `Slew = 0.5Meg` ($0.5\,\mathrm{V/\mu s}$, written in V/s)
* `Ilimit = 20m` (output current limit)
* `Vos = 2m` (input offset voltage)

The one non-ideality UniversalOpAmp2 does not model is the input bias current. You add it yourself: connect a $45\,\mathrm{n A}$ DC current source to each input, exactly as drawn in {numref}`fig-non-ideal-opamp-model`. The configured opamp plus the two current sources **is** the non-ideal opamp model from the Background, now as a working circuit.
```


(Task_S1_4_2)=
#### Task S1: Simulate the bias / offset test circuit
```{admonition} Estimated time: 15 min
:class: estimated-time no-content
```

Build the circuit from [Task A1](Task_A1_4_2) in LTspice around your configured UniversalOpAmp2 **plus the two bias-current sources**. Power it with $\pm 12\,\mathrm{V}$. Tie both inputs to their respective ground or feedback path; do not apply any AC input.

1. Run an operating-point simulation (in LTspice: **Simulate, Edit Simulation Cmd, tab "DC op pnt"**) for $R_a = 1\,\mathrm{k}\Omega$ (the offset-dominant case from Task A1). Record $U_{out}$.
2. Re-run with $R_a = 10\,\mathrm{M}\Omega$ (the bias-dominant case from Task A2). Record $U_{out}$.
3. Convert each $U_{out}$ back to the underlying parameter using the formulas from Task A1. You should recover the $2\,\mathrm{m V}$ and $45\,\mathrm{n A}$ you put into the model. If you do not, the error is in your circuit or in your conversion formula, and finding it now is far cheaper than finding it at the bench.


(Task_S2_4_2)=
#### Task S2: Open-loop and closed-loop frequency response
```{admonition} Estimated time: 15 min
:class: estimated-time no-content
```

Build the non-inverting amplifier around the same configured UniversalOpAmp2. Run an AC analysis (in LTspice: **Simulate, Edit Simulation Cmd, tab "AC Analysis"**, type of sweep Decade, 20 points per decade, from $1\,\mathrm{Hz}$ to $10\,\mathrm{MHz}$). Set the AC amplitude of the input source to 1, so the plotted output is the gain directly.

1. Plot the open-loop gain (with no feedback) as a Bode plot. Mark the unity-gain frequency: where does the curve cross $0\,\mathrm{dB}$? It should sit at the `GBW` value you configured.
2. Plot the closed-loop gain for $G = 2$, $G = 11$, and $G = 101$ (with the resistor values you designed in [Task A3](Task_A3_4_2)) on the same axes. Mark the $-3\,\mathrm{dB}$ point of each. Fill in the simulated column of the table from Task A3.
3. Verify that $G \cdot \mathrm{BW}$ is approximately constant. The first row ($G = 2$) should be a slight outlier because of the "+1" effect.


(Task_S3_4_2)=
#### Task S3: The integrator drifts unless you tame it
```{admonition} Estimated time: 20 min
:class: estimated-time no-content
```

This simulation bridges the bias / offset measurement and the integrator measurement: the same $V_{offset}$ and $I_{bias}$ that produce a few millivolts in Task S1 will, in an integrator, drift the output toward a rail.

1. Build the integrator you designed in [Task A6](Task_A6_4_2) around the configured UniversalOpAmp2 with its two bias-current sources (the answer-box values $R = 47\,\mathrm{k}\Omega$, $C = 100\,\mathrm{n F}$ work). Drive it with the $\pm 2\,\mathrm{V}$, $50\,\mathrm{Hz}$ square wave. Run a transient simulation (in LTspice: **Simulate, Edit Simulation Cmd, tab "Transient"**, stop time $200\,\mathrm{m s}$). Verify the triangle amplitude you predicted.
2. Replace the input source by a $0\,\mathrm{V}$ source and re-run for $60\,\mathrm{s}$. The input side of $R$ is grounded, so part of the inverting input's bias current escapes through $R$, but what remains, together with the amplified offset, still walks the output away. Record approximately how long it takes to reach a rail.
3. Now **delete the source and $R$ entirely**, which is what unplugging the function generator cable does at the bench (an unplugged $R$ carries no current, so removing it changes nothing electrically). The bias current at the inverting input now has one single place to go: into $C$. Predict the drift slope $\lvert \mathrm{d}U_{out}/\mathrm{d}t \rvert = I_{bias}/C$ before you run, then simulate for $60\,\mathrm{s}$ and compare. The drift may be somewhat faster or slower than in step 2 (the offset term is gone and its sign is chip-dependent), but it is now set by $I_{bias}$ alone.
4. Add a "taming" resistor $R_t = 1\,\mathrm{M}\Omega$ in parallel with the feedback capacitor $C$. Repeat steps 2 and 3. The drift should now stop at a small DC value instead of running away, in both cases.
5. Re-run step 1 with the taming resistor in place. The triangle output should be barely distinguishable from the untamed version, because $R_t \gg R$.

The taming resistor is a hands-on demonstration that the **offset and bias** are the cause of the drift, not some mysterious property of the integrator. You will repeat all of this in hardware in [Task I4](Task_I4_4_2) and [Task I5](Task_I5_4_2).


## Practicum

### Implement and investigate

Five measurement blocks are scheduled below. [Task I1](Task_I1_4_2) gives you bias, offset, and the class-wide spread. [Task I2](Task_I2_4_2) gives you GBW and slew rate from one amplifier. [Task I3](Task_I3_4_2) finds the output current limit. [Task I4](Task_I4_4_2) and [Task I5](Task_I5_4_2) are the integrator, first untamed, then tamed.

```{attention}
Bias and offset are millivolt-scale measurements. Use the $200\,\mathrm{m V}$ or $2\,\mathrm{V}$ range on your DMM, not the $20\,\mathrm{V}$ range. The reading should be steady within a few digits; if it jumps, you have a wiring problem (loose lead, missing decoupling cap, or a bias-current path that is not committed).
```

```{figure} images/opamp_DIP8.svg
---
name: fig-opamp-pinout-4-2
height: 400px
---
DIP-8 dual opamp pinout, **top view** (looking down at the chip as it sits in the breadboard), not rotated. The pinout is identical for the LM358P and the TL072CP. You use opamp 1: pin 1 output, pin 2 inverting input, pin 3 non-inverting input, pin 4 $V_- = -12\,\mathrm{V}$, pin 8 $V_+ = +12\,\mathrm{V}$. Pins 5 to 7 belong to opamp 2 and stay unconnected today.
```

```{admonition} Probe the pins before you trust the circuit
:class: tip
Every task in this practicum starts the same way when something looks wrong: put the DMM black lead on ground and measure the DC potential **directly on the opamp pins**, not on the breadboard rails.

* Pin 8 should read $+12\,\mathrm{V}$ and pin 4 should read $-12\,\mathrm{V}$. If not, fix the supply wiring before touching anything else.
* Pin 1 (output) parked near a rail means the opamp is railed: the feedback path is broken, or one of the inputs has no DC path.
* Pins 2 and 3 should sit within millivolts of each other whenever the feedback loop is closed and the output is not railed. If they differ by volts, the virtual short does not hold and none of the golden-rule formulas apply.

Thirty seconds of pin probing beats twenty minutes of blind rewiring.
```


(Task_I1_4_2)=
#### Task I1: Measure $V_{offset}$ and $I_{bias}$, then pool the class results
```{admonition} Estimated time: 30 min
:class: estimated-time no-content
```

One test circuit, two chip parameters, one resistor swap. Build the circuit from [Task A1](Task_A1_4_2) on the ALPACA breadboard: LM358P (opamp 1), $\pm 12\,\mathrm{V}$, $R_f = 1\,\mathrm{M}\Omega$.

**Part 1: offset.** Use the offset-dominant $R_a$ you chose in [Task A1](Task_A1_4_2).

1. Tie the non-inverting input to ground through a resistor equal to the parallel combination of $R_f$ and $R_a$ (this cancels a small bias-current artefact). For your offset-dominant $R_a$, this is approximately $R_a$ itself.
2. Switch on the supply. Read $U_{out}$ on the DMM. Wait 30 s for the reading to stabilise.
3. Convert your measurement to $V_{offset}$ using the formula from Task A1.
4. Note the chip serial number (or just "chip 1", "chip 2"). Different chips give different offsets.

**Part 2: bias.** Without rebuilding anything else, swap $R_a$ for the bias-dominant value you chose in [Task A2](Task_A2_4_2) (a high-megohm resistor; the lab kit has $1\,\mathrm{M}\Omega$ and $10\,\mathrm{M}\Omega$).

5. Read $U_{out}$ on the DMM. The reading is now smaller; expect a few millivolts.
6. Convert to $I_{bias}$ using the formula from Task A1.
7. Compare your $V_{offset}$ and $I_{bias}$ with the LM358 datasheet (max and typical columns).

**Part 3: the class pool.** Enter your $V_{offset}$, $I_{bias}$, and chip number in the interactive class pool; it plots every group's values as they come in.

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">🚩 TODO: build the interactive class pool (shared plot that students submit Voffset / Ibias to) and add its link here</span>

In your notepad, answer:

* What is the spread between the smallest and largest $|V_{offset}|$ in the pool? Does that fit inside the datasheet's "max" column?
* Which has more chip-to-chip variation, $V_{offset}$ or $I_{bias}$?
* If you were buying 100 chips for an instrument, would you trust the typical or the max value of these specs?


(Task_I2_4_2)=
#### Task I2: GBW and slew rate from one amplifier
```{admonition} Estimated time: 50 min
:class: estimated-time no-content
```

Rebuild the LM358 as a non-inverting amplifier on the same breadboard (you can keep the chip in place and change only $R_a$ and $R_b$). The same board delivers both frequency limits of the chip: a **small sine** reveals the bandwidth, a **large square** reveals the slew rate.

**Part 1: GBW from a gain sweep.** Drive the input with the function generator: a sine wave, $50\,\mathrm{m V}$ amplitude, frequency variable. For each row of the table from [Task A3](Task_A3_4_2), with the $R_a$ and $R_b$ values you designed there:

1. Set the gain. Verify the low-frequency gain at $f = 1\,\mathrm{kHz}$ on the scope.
2. Slowly raise the input frequency until the output amplitude has dropped to $0.71$ of the low-frequency value. That frequency is the $-3\,\mathrm{dB}$ point. Record it.
3. Compute $G \cdot \mathrm{BW}$ for that row and fill the table.

```{tip}
Keep the input amplitude small ($50\,\mathrm{m V}$ or so) so the output amplitude is well below the slew-rate limit at the highest gain. If the output starts to look like a triangle as you raise the frequency, you are slew-limited, not bandwidth-limited; lower the input amplitude and try again. The LM358's output stage can also put a small kink in the sine where it crosses zero (crossover distortion); it looks alarming but does not affect the $-3\,\mathrm{dB}$ reading.
```

**Part 2: slew rate from a square wave.** Set the gain to $G = 11$. Drive the input with a square wave, amplitude $1\,\mathrm{V}$, frequency $1\,\mathrm{kHz}$. The output amplitude is $11\,\mathrm{V}$, near the rail.

4. On the scope, zoom on a single rising edge of the output. Use the cursors to measure the slope: $\Delta U / \Delta t$ in V/microsecond.
5. Repeat for the falling edge. Average the two values; this is your measured slew rate.
6. Compare with the LM358 datasheet ($\mathrm{SR} \approx 0.5\,\mathrm{V/\mu s}$), and use your measured SR to re-compute the full power bandwidth from [Task A4](Task_A4_4_2).

```{tip}
Measure the slope between the 10 percent and 90 percent crossings of the output transition, not from corner to corner. The corners include opamp internal delays that do not belong to the slew rate.
```

After both parts, answer in your notepad: is the product $G \cdot \mathrm{BW}$ constant within your measurement accuracy? Which row is the worst fit and why? And write one sentence on why part 1 needed a small amplitude while part 2 needed a large one.


(Task_I3_4_2)=
#### Task I3: Find the output current limit
```{admonition} Estimated time: 15 min
:class: estimated-time no-content
```

Keep the $G = 11$ amplifier from [Task I2](Task_I2_4_2) on the breadboard. Set the input to a $1\,\mathrm{kHz}$ sine of $0.7\,\mathrm{V}$ amplitude, so that the unloaded output is a clean sine of about $8\,\mathrm{V}$ amplitude, safely below the rails.

1. Connect a load resistor $R_L$ from the output to ground. Step it down: $1\,\mathrm{k}\Omega$, $560\,\Omega$, $220\,\Omega$, $100\,\Omega$. Capture the output at each step.
2. As soon as the sine starts to flatten, read the clipping amplitude $\hat U_{clip}$ on the scope and compute $I_{out,max} = \hat U_{clip}/R_L$ for every $R_L$ that clips. The values should agree with each other.
3. Compare with your prediction from [Task A5](Task_A5_4_2) and with the output-current limits in the LM358 datasheet. Do not be surprised if the positive and negative clip levels differ: the LM358 sources and sinks different maximum currents. In your notepad: how do you tell current clipping from rail clipping on the scope?

The LM358 is internally short-circuit protected, so this measurement does not damage the chip. It may get warm; that is the protection circuit doing its job.


(Task_I4_4_2)=
#### Task I4: Build the integrator, watch it drift, scale its slope
```{admonition} Estimated time: 35 min
:class: estimated-time no-content
```

Rebuild the breadboard one more time, this time as the integrator you designed in [Task A6](Task_A6_4_2) ($R = 47\,\mathrm{k}\Omega$, $C = 100\,\mathrm{n F}$ if you use the answer-box values), no taming resistor yet. Drive the input with the $\pm 2\,\mathrm{V}$, $50\,\mathrm{Hz}$ square wave.

**Part 1: the triangle.**

1. On the scope, capture $U_{in}$ on channel 1 and $U_{out}$ on channel 2. Trigger on channel 1.
2. Verify that the output is a triangle. Measure its peak-to-peak amplitude and the slope of one of its ramps. Compare with the prediction from [Task A6](Task_A6_4_2) and the simulation from [Task S3](Task_S3_4_2).

If the triangle is **not centred on zero**, or slowly walks away while you watch, do not panic. That is the offset and bias drift you simulated in [Task S3](Task_S3_4_2), now in real silicon. You will fix it in [Task I5](Task_I5_4_2); first, put it to work.

**Part 2: the drift is a bias-current meter.** Unplug the function generator cable so the input side of $R$ is left floating. The bias current of the inverting input now has exactly one path: into $C$. The output therefore ramps at $|\mathrm{d}U_{out}/\mathrm{d}t| = I_{bias}/C$.

3. Set the scope to a slow timebase ($1\,\mathrm{s/div}$, roll mode if available). Briefly short $C$ with a piece of wire to reset the output to $0\,\mathrm{V}$, release, and record the ramp until it approaches a rail.
4. Measure the ramp slope and compute $I_{bias} = C \cdot |\mathrm{d}U_{out}/\mathrm{d}t|$. Compare with the value from [Task I1](Task_I1_4_2). You have now measured the same chip parameter in two completely different ways; how close are they?

**Part 3: one decade.** Reconnect the square wave. Pick one of the two component changes: $R$ from $47\,\mathrm{k}\Omega$ to $4.7\,\mathrm{k}\Omega$, or $C$ from $100\,\mathrm{n F}$ to $10\,\mathrm{n F}$.

5. Predict the new ramp slope and peak-to-peak amplitude on paper first. Does the predicted amplitude still fit between the rails?
6. Measure. The ramps should be 10 times steeper, but the amplitude cannot follow: the output saturates near the rails and the triangle becomes a clipped trapezoid. Confirm the factor of 10 on the **slope** of the ramp portions, not on the amplitude.


(Task_I5_4_2)=
#### Task I5: Tame the integrator
```{admonition} Estimated time: 20 min
:class: estimated-time no-content
```

Restore your design values ($R = 47\,\mathrm{k}\Omega$, $C = 100\,\mathrm{n F}$) and add a $1\,\mathrm{M}\Omega$ resistor $R_t$ in parallel with $C$ in the feedback path. The integrator is now a "tamed" integrator: at high frequencies it still behaves like an integrator, but at DC its gain is bounded to $-R_t/R \approx -21$ instead of infinity. The bias-current and offset effects can no longer drive the output to a rail.

1. Repeat the two no-input conditions from [Task S3](Task_S3_4_2) and [Task I4](Task_I4_4_2): first with the function generator connected and set to $0\,\mathrm{V}$, then with the cable unplugged. In both cases the output should now settle at a small DC value (of order a hundred millivolts) instead of ramping away.
2. Reconnect the $\pm 2\,\mathrm{V}$, $50\,\mathrm{Hz}$ square wave. The triangle output should look almost identical to [Task I4](Task_I4_4_2). Verify that.
3. Compute the predicted DC level from the formula $U_{out,DC} = (1 + R_t/R)\, V_{offset} + I_{bias} R_t$, using the $V_{offset}$ and $I_{bias}$ you measured in [Task I1](Task_I1_4_2). Compare with what you see on the scope.

This task ties everything in this manual together: the millivolt-scale numbers from the bias / offset measurement explain a real, visible behaviour of a real circuit. That is the whole point of measuring chip non-idealities.


### Compare and conclude

(Task_C1_4_2)=
#### Task C1: Bias and offset, three columns
```{admonition} Consider Tasks [I1](Task_I1_4_2) and [S1](Task_S1_4_2).
:class: note no-content
```

| quantity | datasheet typ | datasheet max | simulated | measured (your chip) | class spread |
|---|---|---|---|---|---|
| $V_{offset}$ |  |  |  |  |  |
| $I_{bias}$ |  |  |  |  |  |

Where does your chip sit? Is the typical column a good guide for design, or do you need to engineer for the max? Did the two independent $I_{bias}$ measurements (DMM in Task I1, drift slope in [Task I4](Task_I4_4_2)) agree?


(Task_C2_4_2)=
#### Task C2: Gain-bandwidth product as a constant
```{admonition} Consider Tasks [A3](Task_A3_4_2), [S2](Task_S2_4_2), [I2](Task_I2_4_2).
:class: note no-content
```

Take the table from Task A3 / S2 / I2 and add one more column: the deviation of $G \cdot \mathrm{BW}$ from its average. Is the product constant within $\pm 20\%$? If the $G = 2$ row deviates more than the others, what is the cause?


(Task_C3_4_2)=
#### Task C3: Re-state the golden rules as inequalities
```{admonition} Consider Tasks [I1](Task_I1_4_2), [I2](Task_I2_4_2), [I3](Task_I3_4_2).
:class: note no-content
```

In Manual 4.1 the golden rules were stated as equalities ($U_+ = U_-$, $I_{in} = 0$, ...). Re-write them as **inequalities** with the numbers you just measured for your chip:

| golden rule (4.1) | tightened version (4.2) |
|---|---|
| $U_+ = U_-$ | $\lvert U_+ - U_- \rvert \le \ldots$ V |
| $I_{in+} = I_{in-} = 0$ | $\lvert I_{in} \rvert \le \ldots$ A |
| the output is an ideal voltage source | $\lvert I_{out} \rvert \le \ldots$ A |
| open-loop gain $\to \infty$ | open-loop gain at frequency $f$ is at most $\ldots$ |
| no rate limit on the output | $\lvert \mathrm{d}U_{out}/\mathrm{d}t \rvert \le \ldots$ V/s |


```{admonition} Deep dive: a design decision
:class: deep-dive, dropdown

You are designing a charge amplifier for a single photodiode in low light. The diode delivers about $100\,\mathrm{p A}$ of signal current. Choose, with one line of justification each:

* the LM358 you measured today, or an opamp whose bias current is far below the $100\,\mathrm{p A}$ signal (FET-input types like your kit's TL072CP sit in the picoamp range)?
* Tamed integrator or untamed integrator?
* Highest-gain stage at the front (close to the diode) or at the back (after a buffer)?

There is no single "right" answer, but every choice should follow from a number you measured this week.

A charge amplifier is an integrator. To feel why the integrator is the noise-friendly choice, compare it against its mirror image on your own breadboard: swap $R$ and $C$, so that $C$ sits at the input and $R$ in the feedback, and you have a **differentiator**. Drive it with a $1\,\mathrm{V}$ amplitude, $100\,\mathrm{Hz}$ sine wave and capture the output. Two things will happen:

* The amplitude of $U_{out}$ scales with frequency (the differentiator is a high-pass filter).
* The output picks up a lot of high-frequency noise that you never saw in the integrator.

The integrator's gain falls with frequency, so it calms high-frequency noise. The differentiator's gain rises with frequency until it runs into the $-20\,\mathrm{dB/decade}$ open-loop roll-off, so it amplifies exactly the noise you do not want. Connect what you see on the scope to that roll-off and to your slew-rate measurement, then write one line on why charge amplifiers integrate rather than differentiate.

For the full theory, see [textbook chapter 7, section on the differentiator](../../../theory_part/7_opamp1.md).

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">🚩 TODO: replace with the published anchor for the differentiator section</span>
```
